# 🔍 Log Parser — Parseur Multi-Format de Fichiers Logs

Ce notebook contient une classe `LogParser` capable de :
- **Parser** automatiquement des fichiers logs bruts (Apache Access, Apache Error, Syslog/Messages)
- **Auto-détecter** le format de chaque fichier
- **Supporter** les fichiers compressés (`.gz`)
- **Reproduire** l'arborescence d'entrée dans le dossier de sortie
- **Exporter** des CSV intermédiaires par fichier source
- **Fusionner** le tout en un CSV consolidé unique

## Importation des bibliothèques

In [13]:
import os
import re
import gzip
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging
from typing import Optional, Dict, List, Tuple

##  Configuration du logging

In [14]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('LogParser')

##  Classe `LogParser`

### Formats supportés

| Format | Champs extraits |
|--------|----------------|
| **Apache Access** (Combined Log) | `ip`, `date`, `method`, `url`, `protocol`, `status_code`, `response_size`, `referer`, `user_agent` |
| **Apache Error** | `date`, `module`, `level`, `pid`, `tid`, `error_code`, `message` |
| **Syslog / Messages** | `date`, `hostname`, `service`, `pid`, `message` |

In [ ]:
# ============================================================
# CELLULE 3 — CLASSE LogParser
# ============================================================

class LogParser:
    """
    Parseur multi-format de fichiers log avec fusion CSV consolidée.

    Formats supportés:
        - Apache Access (Combined Log Format)
        - Apache Error Log
        - Syslog / Messages (DHCP, system)
        - Mail Log (postfix, dovecot, spamd)

    Attributes:
        input_base  (Path): Chemin racine des logs d'entrée
        output_base (Path): Chemin racine des CSV de sortie
        _parsers    (dict): Regex par type de log
        _all_frames (dict): DataFrames parsés {filepath: DataFrame}
    """

    # --------------------------------------------------------
    # PATTERNS REGEX
    # --------------------------------------------------------
    
    #	Patterns regex pour différents types de logs
    # Patterns definit les règles pour decouper les lignes
    #de texte brut en variables distinctes pour 03 formats: Apache Access, Apache Error et Syslog. 
    # Chaque pattern utilise des groupes nommés pour capturer les informations pertinentes de chaque ligne de log.
    _PATTERNS = {
     
        'apache_access': re.compile(
            r'^(?P<ip>[\S:]+)\s+-\s+\S+\s+'
            r'\[(?P<date>[^\]]+)\]\s+'
            r'"(?P<method>\S+)\s+(?P<url>\S+)\s+(?P<protocol>[^"]+)"\s+'
            r'(?P<status_code>\d+)\s+'
            r'(?P<response_size>\S+)\s+'
            r'"(?P<referer>[^"]*)"\s+'
            r'"(?P<user_agent>[^"]*)"'
        ),
        'apache_error': re.compile(
            r'^\[(?P<date>[^\]]+)\]\s+'
            r'\[(?P<module>[^:]+):(?P<level>\w+)\]\s+'
            r'\[pid\s+(?P<pid>\d+)(?::tid\s+(?P<tid>\d+))?\]\s+'
            r'(?:(?P<error_code>\w+\d+):\s+)?'
            r'(?P<message>.+)$'
        ),
        'syslog': re.compile(
            r'^(?P<date>\w+\s+\d+\s+\d+:\d+:\d+)\s+'
            r'(?P<hostname>\S+)\s+'
            r'(?P<service>[^\[\s:]+)(?:\[(?P<pid>\d+)\])?:\s+'
            r'(?P<message>.+)$'
        ),
    }

    # Extensions de fichiers reconnues comme logs
    _LOG_EXTENSIONS = {
        '', '.log', '.gz', '.1', '.2', '.3', '.4', '.5'
    }

    # --------------------------------------------------------
    # CONSTRUCTEUR
    # --------------------------------------------------------
    def __init__(self, input_base: str, output_base: str):
        """
        Initialise le parseur.

        Args:
            input_base  (str): Dossier racine contenant les logs (ex: 'data/')
            output_base (str): Dossier racine pour les CSV de sortie (ex: 'logs_output/')
        """
        self.input_base  = Path(input_base)
        self.output_base = Path(output_base)
        self._parsers    = self._PATTERNS
        self._all_frames: Dict[Path, pd.DataFrame] = {}

        # Créer le dossier de sortie
        self.output_base.mkdir(parents=True, exist_ok=True)

        logger.info(f'LogParser initialisé')
        logger.info(f'  Entrée : {self.input_base}')
        logger.info(f'  Sortie : {self.output_base}')

    # --------------------------------------------------------
    # LECTURE DE FICHIER (.gz et texte)
    # --------------------------------------------------------
    def _read_file(self, filepath: Path) -> List[str]:
        """
        Lit un fichier log (texte brut ou compressé .gz).

        Args:
            filepath (Path): Chemin du fichier

        Returns:
            list[str]: Liste des lignes
        """
        try:
            if filepath.suffix == '.gz':
                with gzip.open(filepath, 'rt', encoding='utf-8', errors='replace') as f:
                    return f.readlines()
            else:
                with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
                    return f.readlines()
        except Exception as e:
            logger.warning(f'Impossible de lire {filepath.name}: {e}')
            return []

    # --------------------------------------------------------
    # AUTO-DÉTECTION DU FORMAT
    # --------------------------------------------------------
    def _detect_log_type(self, filepath: Path) -> str:
        """
        Détecte automatiquement le type de log en lisant les 10 premières lignes.

        Stratégie:
            - Teste chaque regex sur les premières lignes non vides
            - Retourne le type qui matche le plus souvent
            - Fallback sur 'syslog' si aucun match

        Args:
            filepath (Path): Chemin du fichier

        Returns:
            str: 'apache_access', 'apache_error', ou 'syslog'
        """
        lines = self._read_file(filepath)

        # Prendre les 10 premières lignes non vides
        sample = [l.strip() for l in lines if l.strip()][:10]

        if not sample:
            return 'syslog'  # Fallback

        # Compter les matches par type
        scores = {log_type: 0 for log_type in self._parsers}
        for line in sample:
            for log_type, pattern in self._parsers.items():
                if pattern.match(line):
                    scores[log_type] += 1

        # Retourner le type dominant
        best_type = max(scores, key=scores.get)
        if scores[best_type] == 0:
            return 'syslog'  # Fallback

        return best_type

    # --------------------------------------------------------
    # PARSEURS PAR FORMAT
    # --------------------------------------------------------
    def _parse_apache_access(self, line: str) -> Optional[dict]:
        """
        Parse une ligne Apache Combined Log Format.

        Format:
            IP - - [DATE] "METHOD URL PROTOCOL" STATUS SIZE "REFERER" "USER_AGENT"

        Args:
            line (str): Ligne brute

        Returns:
            dict | None: Champs extraits ou None si pas de match
        """
        match = self._parsers['apache_access'].match(line.strip())
        if not match:
            return None
           
           ## record sert à stocker les informations extraites de la ligne de log après avoir été analysée par l'expression régulière. 
           # Il est initialisé en utilisant la méthode `groupdict()` de l'objet `match`, 
           # qui retourne un dictionnaire contenant les groupes nommés capturés par l'expression régulière. 
           # Chaque clé du dictionnaire correspond à un nom de groupe défini dans le pattern regex, 
           # et chaque valeur correspond à la partie de la ligne de log qui a été capturée pour ce groupe.

        record = match.groupdict()

        # Parser le timestamp
        try:
            record['datetime'] = pd.to_datetime(
                record['date'], format='%d/%b/%Y:%H:%M:%S %z'
            )
        except Exception:
            record['datetime'] = pd.NaT

        # Convertir types numériques
        record['status_code']    = int(record['status_code'])
        record['response_size']  = pd.to_numeric(record['response_size'], errors='coerce')

        # Nettoyage
        record['referer']    = None if record['referer']    == '-' else record['referer']
        record['user_agent'] = None if record['user_agent'] == '-' else record['user_agent']

        return record

    def _parse_apache_error(self, line: str) -> Optional[dict]:
        """
        Parse une ligne Apache Error Log.

        Format:
            [DATE] [MODULE:LEVEL] [pid PID:tid TID] ERROR_CODE: MESSAGE

        Args:
            line (str): Ligne brute

        Returns:
            dict | None: Champs extraits ou None si pas de match
        """
        match = self._parsers['apache_error'].match(line.strip())
        if not match:
            return None

        record = match.groupdict()

        # Parser le timestamp
        try:
            record['datetime'] = pd.to_datetime(
                record['date'], format='%a %b %d %H:%M:%S.%f %Y'
            )
        except Exception:
            try:
                record['datetime'] = pd.to_datetime(record['date'])
            except Exception:
                record['datetime'] = pd.NaT

        # Convertir pid/tid
        if record.get('pid'):
            record['pid'] = int(record['pid'])
        if record.get('tid'):
            record['tid'] = int(record['tid'])

        return record

    def _parse_syslog(self, line: str) -> Optional[dict]:
        """
        Parse une ligne Syslog (DHCP, mail, system).

        Format:
            MONTH DAY TIME HOSTNAME SERVICE[PID]: MESSAGE

        Args:
            line (str): Ligne brute

        Returns:
            dict | None: Champs extraits ou None si pas de match
        """
        match = self._parsers['syslog'].match(line.strip())
        if not match:
            return None

        record = match.groupdict()

        # Parser le timestamp (ajouter l'année courante)
        try:
            year = datetime.now().year
            record['datetime'] = pd.to_datetime(
                f"{year} {record['date']}", format='%Y %b %d %H:%M:%S'
            )
        except Exception:
            record['datetime'] = pd.NaT

        # Convertir pid
        if record.get('pid'):
            record['pid'] = int(record['pid'])

        return record

    # --------------------------------------------------------
    # PARSE UN FICHIER COMPLET
    # --------------------------------------------------------
    def parse_file(self, filepath: Path) -> pd.DataFrame:
        """
        Parse un fichier log complet en DataFrame.

        Étapes:
            1. Détecter le format du fichier
            2. Lire toutes les lignes
            3. Parser chaque ligne avec le bon parseur
            4. Ajouter colonnes de traçabilité (source_file, log_type)

        Args:
            filepath (Path): Chemin du fichier

        Returns:
            pd.DataFrame: DataFrame des lignes parsées
        """
        filepath = Path(filepath)

        # Détecter le type de log
        log_type = self._detect_log_type(filepath)

        # Choisir le bon parseur
        parseur = {
            'apache_access': self._parse_apache_access,
            'apache_error':  self._parse_apache_error,
            'syslog':        self._parse_syslog,
        }[log_type]

        # Lire le fichier
        lines = self._read_file(filepath)

        if not lines:
            logger.warning(f'  ⚠ Fichier vide ou illisible: {filepath.name}')
            return pd.DataFrame()

        # Parser chaque ligne
        records = []
        skipped = 0
        for line in lines:
            record = parseur(line)
            if record:
                record['source_file'] = filepath.name
                record['log_type']    = log_type
                records.append(record)
            else:
                skipped += 1

        if not records:
            logger.warning(f'  ⚠ Aucune ligne parsée: {filepath.name}')
            return pd.DataFrame()

        df = pd.DataFrame(records)
        logger.info(
            f'  ✓ {filepath.name} [{log_type}] '
            f'→ {len(df):,d} lignes parsées, {skipped} ignorées'
        )
        return df

    # --------------------------------------------------------
    # CHEMIN DE SORTIE EN MIROIR
    # --------------------------------------------------------
    def _mirror_output_path(self, input_path: Path) -> Path:
        """
        Calcule le chemin de sortie CSV en miroir de l'arborescence d'entrée.

        Exemple:
            Entrée : data/DHCP/messages_log/messages
            Sortie : logs_output/DHCP/messages_log/messages.csv

        Args:
            input_path (Path): Chemin absolu du fichier d'entrée

        Returns:
            Path: Chemin absolu du CSV de sortie
        """
        # Chemin relatif depuis input_base
        relative = input_path.relative_to(self.input_base)

        # Construire chemin miroir + extension .csv
        output_path = self.output_base / relative.parent / (relative.name + '.csv')

        return output_path

    # --------------------------------------------------------
    # SCAN ET PARSE DE TOUS LES FICHIERS
    # --------------------------------------------------------
    def scan_and_parse(self, subdirs: Optional[List[str]] = None):
        """
        Scanne récursivement input_base et parse tous les fichiers log trouvés.

        Args:
            subdirs (list[str] | None): Limiter à certains sous-dossiers.
                                        None = scanner tout input_base.

        Exemple:
            parser.scan_and_parse()                        # Tout scanner
            parser.scan_and_parse(['DHCP', 'messagerie'])  # Seulement ces dossiers
        """
        # Déterminer les dossiers à scanner
        if subdirs:
            roots = [self.input_base / s for s in subdirs]
        else:
            roots = [self.input_base]

        logger.info(f' Scan de {self.input_base} ...')
        total_files   = 0
        total_records = 0

        for root in roots:
            if not root.exists():
                logger.warning(f'  Dossier introuvable: {root}')
                continue

            for filepath in sorted(root.rglob('*')):
                # Ignorer dossiers et fichiers Python
                if not filepath.is_file():
                    continue
                if filepath.suffix in {'.py', '.ipynb', '.md', '.csv'}:
                    continue

                # Ignorer fichiers trop petits (< 10 octets)
                if filepath.stat().st_size < 10:
                    logger.warning(f'  ⚠ Fichier vide ignoré: {filepath.name}')
                    continue

                # Parser le fichier
                logger.info(f'📄 {filepath.relative_to(self.input_base)}')
                df = self.parse_file(filepath)

                if not df.empty:
                    self._all_frames[filepath] = df
                    total_files   += 1
                    total_records += len(df)

        logger.info(f'\n Scan terminé: {total_files} fichiers, {total_records:,d} enregistrements')

    # --------------------------------------------------------
    # EXPORT CSV INTERMÉDIAIRES (arborescence miroir)
    # --------------------------------------------------------
    def export_intermediate_csv(self):
        """
        Exporte un CSV par fichier source dans l'arborescence miroir.

        Structure de sortie:
            logs_output/
            ├── DHCP/messages_log/messages.csv
            ├── messagerie/mail_log/mail.log.1.csv
            └── ...
        """
        if not self._all_frames:
            logger.warning('Aucune donnée à exporter. Appelez scan_and_parse() d\'abord.')
            return

        logger.info(f' Export des CSV intermédiaires...')
        exported = 0

        for input_path, df in self._all_frames.items():
            output_path = self._mirror_output_path(input_path)

            # Créer les dossiers intermédiaires
            output_path.parent.mkdir(parents=True, exist_ok=True)

            # Exporter
            df.to_csv(output_path, index=False)
            exported += 1
            logger.info(f'  ✓ {output_path.relative_to(self.output_base)} ({len(df):,d} lignes)')

        logger.info(f'\n  {exported} CSV exportés dans {self.output_base}')

    # --------------------------------------------------------
    # FUSION EN CSV CONSOLIDÉ
    # --------------------------------------------------------
    def fuse(self, output_name: str = 'Combined_All_Logs.csv') -> pd.DataFrame:
        """
        Fusionne tous les DataFrames parsés en un seul CSV consolidé.

        Le CSV final contient toutes les colonnes possibles.
        Les champs non applicables à un type de log sont NaN.
        Une colonne 'log_type' identifie le format de chaque ligne.

        Args:
            output_name (str): Nom du fichier CSV consolidé

        Returns:
            pd.DataFrame: DataFrame fusionné
        """
        if not self._all_frames:
            logger.warning('Aucune donnée à fusionner. Appelez scan_and_parse() d\'abord.')
            return pd.DataFrame()

        logger.info(' Fusion de tous les DataFrames...')

        # Concaténer tous les DataFrames (colonnes différentes → NaN)
        all_dfs = list(self._all_frames.values())
        combined = pd.concat(all_dfs, ignore_index=True, sort=False)

        # Trier par datetime
        if 'datetime' in combined.columns:
            combined = combined.sort_values('datetime', na_position='last')
            combined = combined.reset_index(drop=True)

        # Exporter
        output_path = self.output_base / output_name
        combined.to_csv(output_path, index=False)

        size_mb = output_path.stat().st_size / (1024 * 1024)
        logger.info(f'\n CSV consolidé: {output_path}')
        logger.info(f'   Lignes    : {len(combined):,d}')
        logger.info(f'   Colonnes  : {len(combined.columns)}')
        logger.info(f'   Taille    : {size_mb:.2f} MB')

        return combined

    # --------------------------------------------------------
    # RAPPORT FINAL
    # --------------------------------------------------------
    def summary(self):
        """
        Affiche un résumé complet des fichiers traités et des résultats.
        """
        if not self._all_frames:
            print('Aucune donnée. Appelez scan_and_parse() d\'abord.')
            return

        print('\n' + '='*70)
        print(' RÉSUMÉ — LogParser')
        print('='*70)
        print(f'  Entrée  : {self.input_base}')
        print(f'  Sortie  : {self.output_base}')
        print(f'  Fichiers parsés  : {len(self._all_frames)}')

        # Totaux
        total = sum(len(df) for df in self._all_frames.values())
        print(f'  Total enregistrements : {total:,d}')

        # Distribution par type de log
        print('\n  Distribution par type:')
        type_counts = {}
        for df in self._all_frames.values():
            if 'log_type' in df.columns:
                for log_type, count in df['log_type'].value_counts().items():
                    type_counts[log_type] = type_counts.get(log_type, 0) + count

        for log_type, count in sorted(type_counts.items()):
            pct = count / total * 100 if total > 0 else 0
            print(f'    {log_type:<20}: {count:>8,d} ({pct:5.1f}%)')

        # Détail par fichier
        print('\n  Détail par fichier:')
        print(f'  {"Fichier":<45} {"Type":<15} {"Lignes":>8}')
        print(f'  {"-"*45} {"-"*15} {"-"*8}')
        for input_path, df in sorted(self._all_frames.items()):
            rel   = str(input_path.relative_to(self.input_base))
            ltype = df['log_type'].iloc[0] if 'log_type' in df.columns else '?'
            print(f'  {rel:<45} {ltype:<15} {len(df):>8,d}')

        print('='*70 + '\n')


print('✅ Classe LogParser définie')

✅ Classe LogParser définie


## Exécution

Modifiez les chemins ci-dessous si nécessaire, puis exécutez la cellule.

In [ ]:
# ═══ CONFIGURATION DES CHEMINS ═══
INPUT_BASE  = r"C:\Users\DELL\Desktop\projet\data\Ancien_Site web\httpd\en-access_log"
OUTPUT_BASE = r""

# ═══ INITIALISATION DU PARSEUR ═══
parser = LogParser(INPUT_BASE, OUTPUT_BASE)

# 1. Scanner et parser tous les fichiers logs
parser.scan_and_parse()

# 2. Exporter les CSV intermédiaires (arborescence miroir)
parser.export_intermediate_csv()

# 3. Fusionner en un CSV consolidé
combined = parser.fuse("Combined_All_Logs.csv")

# 4. Résumé
parser.summary()

2026-07-30 17:16:58,180 - LogParser - INFO - LogParser initialisé
2026-07-30 17:16:58,182 - LogParser - INFO -   Entrée : C:\Users\DELL\Desktop\projet\data\DHCP\messages_log
2026-07-30 17:16:58,185 - LogParser - INFO -   Sortie : C:\Users\DELL\Desktop\projet\logs_output\DHCP\messages.csv
2026-07-30 17:16:58,187 - LogParser - INFO -  Scan de C:\Users\DELL\Desktop\projet\data\DHCP\messages_log ...
2026-07-30 17:16:58,191 - LogParser - INFO - 📄 messages


KeyboardInterrupt: 

## Aperçu rapide du CSV consolidé

In [ ]:
# Aperçu des premières lignes
print(f"Shape: {combined.shape}")
print(f"Colonnes: {list(combined.columns)}")
print(f"\nTypes de logs présents: {combined['log_type'].value_counts().to_dict()}")
combined.head(10)

Shape: (4199, 12)
Colonnes: ['ip', 'date', 'method', 'url', 'protocol', 'status_code', 'response_size', 'referer', 'user_agent', 'datetime', 'source_file', 'log_type']

Types de logs présents: {'apache_access': 4199}


,ip,date,method,url,protocol,status_code,response_size,referer,user_agent,datetime,source_file,log_type
0,207.46.13.44,18/Jul/2021:04:05:37 +0000,GET,/robots.txt,HTTP/1.1,404,208.0,NaN,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,2021-07-18 04:05:37+00:00,erasmustogo-access.log-20210725,apache_access
1,207.46.13.78,18/Jul/2021:04:05:49 +0000,GET,/Website%20documents/Appel%20a%20Propositions%...,HTTP/1.1,304,NaN,NaN,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,2021-07-18 04:05:49+00:00,erasmustogo-access.log-20210725,apache_access
2,213.180.203.46,18/Jul/2021:10:29:17 +0000,GET,/robots.txt,HTTP/1.1,404,208.0,NaN,Mozilla/5.0 (compatible; YandexBot/3.0; +http:...,2021-07-18 10:29:17+00:00,erasmustogo-access.log-20210725,apache_access
3,213.180.203.46,18/Jul/2021:10:29:20 +0000,GET,/,HTTP/1.1,200,81075.0,NaN,Mozilla/5.0 (compatible; YandexBot/3.0; +http:...,2021-07-18 10:29:20+00:00,erasmustogo-access.log-20210725,apache_access
4,94.23.219.171,18/Jul/2021:13:36:47 +0000,GET,/Website%20documents/Newsletters/Erasmus%20UPT...,HTTP/1.0,200,1178420.0,https://erasmustogo.univ-lome.tg/Website%20doc...,Mozilla/5.0 (Windows NT 6.1; Win64; x64) Apple...,2021-07-18 13:36:47+00:00,erasmustogo-access.log-20210725,apache_access
5,94.23.219.171,18/Jul/2021:13:36:48 +0000,GET,/,HTTP/1.0,200,81075.0,https://erasmustogo.univ-lome.tg/,Mozilla/5.0 (Windows NT 6.1; Win64; x64) Apple...,2021-07-18 13:36:48+00:00,erasmustogo-access.log-20210725,apache_access
6,23.82.193.102,18/Jul/2021:15:06:45 +0000,GET,/Website%20documents/Newsletters/Erasmus%20UPT...,HTTP/1.0,200,1178420.0,https://erasmustogo.univ-lome.tg/Website%20doc...,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2021-07-18 15:06:45+00:00,erasmustogo-access.log-20210725,apache_access
7,23.82.193.102,18/Jul/2021:15:06:49 +0000,GET,/,HTTP/1.0,200,81075.0,https://erasmustogo.univ-lome.tg/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2021-07-18 15:06:49+00:00,erasmustogo-access.log-20210725,apache_access
8,66.249.66.221,18/Jul/2021:15:39:13 +0000,GET,/robots.txt,HTTP/1.1,404,208.0,NaN,Mozilla/5.0 (compatible; Googlebot/2.1; +http:...,2021-07-18 15:39:13+00:00,erasmustogo-access.log-20210725,apache_access
9,66.249.66.223,18/Jul/2021:15:39:14 +0000,GET,/index.html,HTTP/1.1,304,NaN,NaN,Mozilla/5.0 (Linux; Android 6.0.1; Nexus 5X Bu...,2021-07-18 15:39:14+00:00,erasmustogo-access.log-20210725,apache_access
